In [1]:
"""
Tabular Q-learning example on FrozenLake (discrete environment).
Dependencies:
  pip install gym==0.26.5 numpy
(If gym version mismatches, adapt env creation accordingly)
"""
import gym
import numpy as np
import random
from collections import defaultdict

def q_learning(env, episodes=2000, alpha=0.85, gamma=0.99, epsilon=1.0, eps_min=0.01, eps_decay=0.995):
    # Q-table as a dict of arrays (useful if observation space isn't 0..N-1)
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    rewards_history = []

    for ep in range(episodes):
        state, _ = env.reset()  # for gym>=0.26, reset returns (state, info)
        done = False
        total_reward = 0.0

        while not done:
            # epsilon-greedy
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            # Q update
            old_value = Q[state][action]
            next_max = np.max(Q[next_state])
            Q[state][action] = old_value + alpha * (reward + gamma * next_max - old_value)

            state = next_state
            total_reward += reward

        # decay exploration
        epsilon = max(eps_min, epsilon * eps_decay)
        rewards_history.append(total_reward)

        if (ep + 1) % 200 == 0:
            avg = np.mean(rewards_history[-200:])
            print(f"Episode {ep+1}/{episodes}  avg_reward(last200)={avg:.3f}  eps={epsilon:.3f}")

    return Q, rewards_history

if __name__ == "__main__":
    env = gym.make("FrozenLake-v1", is_slippery=False)  # deterministic for easier learning
    Q, rewards = q_learning(env, episodes=2000)

    # Demonstrate learned policy
    state, _ = env.reset()
    done = False
    env.render()
    print("Demonstration run:")
    while not done:
        action = int(np.argmax(Q[state]))
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        env.render()
    print("Finished demo")

ModuleNotFoundError: No module named 'gym'